# Data Cleaing and Transformation Process

In [1]:
#importing libraies

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# import the dataset
data = pd.read_csv("../data/healthcare_dataset.csv")
data.head()

In [3]:
# Check the duplicat values 
data.duplicated().sum()

In [4]:
# Drop the duplicated rows 
data = data.drop_duplicates()
data.duplicated().sum()

droping the duplicate rows as it creates a wrong insight 

In [5]:
# columns
data.columns

In [6]:
# check the unique value in Gender
data['Gender'].unique()

In [7]:
# let's confirm the date of admission and discharge date 

print(data['Date of Admission'].dtype)
print(data['Discharge Date'].dtype)

str
str


In [8]:
# Change the datatype of both object that needed to be in datetime 

data['Date of Admission'] =  pd.to_datetime(data['Date of Admission'])
data['Discharge Date'] =  pd.to_datetime(data['Discharge Date'])


print(data['Date of Admission'].dtype)
print(data['Discharge Date'].dtype)

datetime64[us]
datetime64[us]


In [9]:
#let's check the sanity range do we have 


print(data['Date of Admission'].min())
print(data['Date of Admission'].max())

2019-05-08 00:00:00
2024-05-07 00:00:00


### Finding the length of stay 

length of stay is imp because in healcare , longer stay means higher the cost and lower bed avilibility to other new patients 

In [10]:
data['Length of stay'] = (data['Discharge Date'] - data['Date of Admission']).dt.days
print(data['Length of stay'].describe())

count    54966.000000
mean        15.499290
std          8.661471
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: Length of stay, dtype: float64


In [11]:
# Sanity check — any negative stays? (patient discharged before admitted?)
print('\nNegative Length of Stay:', (data['Length of stay'] < 0).sum())

# Preview
print(data[['Date of Admission', 'Discharge Date', 'Length of stay']].head())


Negative Length of Stay: 0
  Date of Admission Discharge Date  Length of stay
0        2024-01-31     2024-02-02               2
1        2019-08-20     2019-08-26               6
2        2022-09-22     2022-10-07              15
3        2020-11-18     2020-12-18              30
4        2022-09-19     2022-10-09              20


  Average 15 days is quite long. A hospital administrator would immediately ask which conditions are driving the longer stays? Can we reduce them? 

In [12]:
data.columns

In [13]:
# Standardize the text column
print(data['Name'].head(10).tolist())

['Bobby JacksOn', 'LesLie TErRy', 'DaNnY sMitH', 'andrEw waTtS', 'adrIENNE bEll', 'EMILY JOHNSOn', 'edwArD EDWaRDs', 'CHrisTInA MARtinez', 'JASmINe aGuIlaR', 'ChRISTopher BerG']


In [14]:
# Fix person/hospital name formatting and trim whitespace
data['Name'] = data['Name'].astype(str).str.strip().str.title()
data['Doctor'] = data['Doctor'].astype(str).str.strip().str.title()
data['Hospital'] = data['Hospital'].astype(str).str.strip().str.title()

data[['Name', 'Doctor', 'Hospital']].head()


In [15]:
# Normalize categorical text values
canonical_maps = {
    'Gender': {'male': 'Male', 'female': 'Female'},
    'Admission Type': {'elective': 'Elective', 'urgent': 'Urgent', 'emergency': 'Emergency'},
    'Medical Condition': {
        'arthritis': 'Arthritis', 'asthma': 'Asthma', 'cancer': 'Cancer',
        'diabetes': 'Diabetes', 'hypertension': 'Hypertension', 'obesity': 'Obesity'
    },
    'Medication': {
        'aspirin': 'Aspirin', 'ibuprofen': 'Ibuprofen', 'lipitor': 'Lipitor',
        'paracetamol': 'Paracetamol', 'penicillin': 'Penicillin'
    },
    'Test Results': {'normal': 'Normal', 'abnormal': 'Abnormal', 'inconclusive': 'Inconclusive'},
    'Insurance Provider': {
        'aetna': 'Aetna', 'blue cross': 'Blue Cross', 'cigna': 'Cigna',
        'medicare': 'Medicare', 'unitedhealthcare': 'UnitedHealthcare'
    }
}

for col in canonical_maps:
    data[col] = data[col].astype(str).str.strip()
    data[col] = data[col].str.lower().map(canonical_maps[col]).fillna(data[col])

data['Blood Type'] = data['Blood Type'].astype(str).str.strip().str.upper()

for col in ['Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Admission Type', 'Medication', 'Test Results']:
    print(f'{col}:', sorted(data[col].unique().tolist()))


Gender: ['Female', 'Male']
Blood Type: ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']
Medical Condition: ['Arthritis', 'Asthma', 'Cancer', 'Diabetes', 'Hypertension', 'Obesity']
Insurance Provider: ['Aetna', 'Blue Cross', 'Cigna', 'Medicare', 'UnitedHealthcare']
Admission Type: ['Elective', 'Emergency', 'Urgent']
Medication: ['Aspirin', 'Ibuprofen', 'Lipitor', 'Paracetamol', 'Penicillin']
Test Results: ['Abnormal', 'Inconclusive', 'Normal']


In [16]:
# Data quality checks for numeric and date fields
print('Missing values total:', int(data.isna().sum().sum()))
print('Negative length of stay:', int((data['Length of stay'] < 0).sum()))
print('Negative billing amount:', int((data['Billing Amount'] < 0).sum()))
print('Age outside 0-120:', int(((data['Age'] < 0) | (data['Age'] > 120)).sum()))

data[['Age', 'Billing Amount', 'Room Number', 'Length of stay']].describe()


Missing values total: 0
Negative length of stay: 0
Negative billing amount: 106
Age outside 0-120: 0


### Handling invalid billing values

Some rows contain negative billing values, which are not valid in this context.
We replace only those negatives with a robust median based on medical condition and admission type.


In [17]:
# Replace negative billing amounts with grouped medians
negative_mask = data['Billing Amount'] < 0
print('Rows with negative billing before fix:', int(negative_mask.sum()))

group_medians = data.loc[~negative_mask].groupby(['Medical Condition', 'Admission Type'])['Billing Amount'].median()
global_median = data.loc[~negative_mask, 'Billing Amount'].median()

for idx in data.index[negative_mask]:
    key = (data.at[idx, 'Medical Condition'], data.at[idx, 'Admission Type'])
    data.at[idx, 'Billing Amount'] = group_medians.get(key, global_median)

print('Rows with negative billing after fix:', int((data['Billing Amount'] < 0).sum()))


Rows with negative billing before fix: 106
Rows with negative billing after fix: 0


In [18]:
# Additional transformed features for analysis
data['Admission Year'] = data['Date of Admission'].dt.year
data['Admission Month'] = data['Date of Admission'].dt.month
data['Admission Day Name'] = data['Date of Admission'].dt.day_name()
data['Billing Category'] = pd.qcut(data['Billing Amount'], q=4, labels=['Low', 'Mid-Low', 'Mid-High', 'High'])

data[['Admission Year', 'Admission Month', 'Admission Day Name', 'Billing Category']].head()


In [19]:
# Final validation and export cleaned dataset
cleaned_path = '../data/healthcare_dataset_cleaned.csv'
data.to_csv(cleaned_path, index=False)

print('Cleaned shape:', data.shape)
print('Remaining duplicates:', int(data.duplicated().sum()))
print('Remaining missing values:', int(data.isna().sum().sum()))
print('Negative billing rows remaining:', int((data['Billing Amount'] < 0).sum()))
print('Saved cleaned file to:', cleaned_path)


Cleaned shape: (54966, 20)
Remaining duplicates: 0
Remaining missing values: 0
Negative billing rows remaining: 0
Saved cleaned file to: ../data/healthcare_dataset_cleaned.csv
